[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/EN/Seminar_Notebooks/chapter5b_multivariate_garch_seminar_notebook.ipynb)

---

# Seminar: Multivariate GARCH Models - Practice Exercises

**Course:** Time Series Analysis and Forecasting  
**Program:** Bachelor program, Faculty of Cybernetics, Statistics and Economic Informatics, Bucharest University of Economic Studies, Romania  
**Academic Year:** 2025-2026

---

## Exercises Overview

1. **Exercise 1:** Rolling Correlations and Stylized Facts
2. **Exercise 2:** CCC vs DCC Estimation on Real Data
3. **Exercise 3:** Portfolio VaR with Dynamic Covariance
4. **Exercise 4:** Dynamic Hedging Ratios
5. **Exercise 5:** Bitcoin-Ethereum Multivariate Analysis

## Setup

In [ ]:
# Install packages if needed (for Colab)
try:
    from arch import arch_model
    import yfinance as yf
except ImportError:
    !pip install arch yfinance --quiet
    from arch import arch_model
    import yfinance as yf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

from arch import arch_model
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf

# Plotting style
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['legend.frameon'] = False

COLORS = {'blue': '#1A3A6E', 'red': '#DC3545', 'green': '#2E7D32',
          'orange': '#E67E22', 'purple': '#8E44AD', 'gray': '#666666'}

# DCC helper functions
def estimate_dcc(z, Qbar, x0=[0.02, 0.95]):
    """Estimate DCC parameters (a, b) from standardized residuals."""
    T, N = z.shape
    
    def neg_loglik(params):
        a, b = params
        Qt = Qbar.copy()
        ll = 0.0
        for t in range(1, T):
            Qt = (1 - a - b) * Qbar + a * np.outer(z[t-1], z[t-1]) + b * Qt
            d = np.sqrt(np.diag(Qt))
            Rt = Qt / np.outer(d, d)
            sign, logdet = np.linalg.slogdet(Rt)
            if sign <= 0:
                return 1e10
            Rt_inv = np.linalg.inv(Rt)
            ll += logdet + z[t] @ Rt_inv @ z[t] - z[t] @ z[t]
        return 0.5 * ll
    
    result = minimize(neg_loglik, x0=x0, method='SLSQP',
                      bounds=[(1e-6, 0.3), (1e-6, 0.9999)],
                      constraints={'type': 'ineq', 'fun': lambda p: 0.9999 - p[0] - p[1]})
    return result.x

def dcc_correlations(z, Qbar, a, b):
    """Compute DCC time-varying correlation matrix series."""
    T, N = z.shape
    Rt_series = np.zeros((T, N, N))
    Qt = Qbar.copy()
    Rt_series[0] = Qbar / np.outer(np.sqrt(np.diag(Qbar)), np.sqrt(np.diag(Qbar)))
    
    for t in range(1, T):
        Qt = (1 - a - b) * Qbar + a * np.outer(z[t-1], z[t-1]) + b * Qt
        d = np.sqrt(np.diag(Qt))
        Rt_series[t] = Qt / np.outer(d, d)
    return Rt_series

print("Setup complete! Ready to analyze multivariate volatility.")

---

## Exercise 1: Rolling Correlations and Stylized Facts

**Objective:** Demonstrate that correlations are not constant using real financial data.

**Tasks:**
1. Download S&P 500 and FTSE 100 data
2. Compute rolling correlations with different window sizes
3. Show that correlations increase during crises

[QuantLet: TSA_ch5b_rolling_corr](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_rolling_corr)

In [ ]:
# Task 1: Download S&P 500 and FTSE 100
print("Downloading S&P 500 and FTSE 100 data...")

sp500 = yf.download('^GSPC', start='2005-01-01', end='2024-12-31', progress=False)
ftse = yf.download('^FTSE', start='2005-01-01', end='2024-12-31', progress=False)

r_sp = (sp500['Close'].squeeze().pct_change() * 100).dropna()
r_ftse = (ftse['Close'].squeeze().pct_change() * 100).dropna()

returns = pd.concat([r_sp, r_ftse], axis=1, join='inner').dropna()
returns.columns = ['SP500', 'FTSE100']

print(f"\nData: {returns.index[0].date()} to {returns.index[-1].date()}, {len(returns)} obs")
print(f"\nFull-sample correlation: {returns.corr().iloc[0, 1]:.4f}")
print(f"Skewness: SP500={stats.skew(returns['SP500']):.3f}, FTSE={stats.skew(returns['FTSE100']):.3f}")
print(f"Kurtosis: SP500={stats.kurtosis(returns['SP500'])+3:.2f}, FTSE={stats.kurtosis(returns['FTSE100'])+3:.2f}")

In [ ]:
# Task 2: Rolling correlations with different windows
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

for window, color, alpha in [(30, COLORS['orange'], 0.5), (60, COLORS['blue'], 0.7), 
                               (120, COLORS['green'], 0.9)]:
    rc = returns['SP500'].rolling(window).corr(returns['FTSE100'])
    axes[0].plot(returns.index, rc, color=color, alpha=alpha, linewidth=0.8,
                label=f'{window}-day window')

axes[0].axhline(y=returns.corr().iloc[0, 1], color=COLORS['red'], linestyle='--',
                linewidth=1.5, label=f'Full-sample: {returns.corr().iloc[0, 1]:.3f}')
axes[0].set_ylabel('Correlation')
axes[0].set_title('Rolling Correlations: S&P 500 vs FTSE 100', fontweight='bold')
axes[0].set_ylim(-0.2, 1.0)
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=4)

# Task 3: Crisis vs calm comparison
crisis_2008 = returns.loc['2008-09':'2009-03']
calm_2013 = returns.loc['2013-01':'2013-06']
covid = returns.loc['2020-02':'2020-04']

periods = [('2008 Crisis', crisis_2008, COLORS['red']),
           ('2013 Calm', calm_2013, COLORS['green']),
           ('COVID-19', covid, COLORS['orange'])]

bp_data = [p[1].corr().iloc[0, 1] for p in periods]  # Single correlation per period
# Use scatter plot of returns to show correlation visually
for i, (name, data, color) in enumerate(periods):
    axes[1].scatter(data['SP500'], data['FTSE100'], color=color, alpha=0.5, s=15, label=name)

axes[1].set_xlabel('S&P 500 Return (%)')
axes[1].set_ylabel('FTSE 100 Return (%)')
axes[1].set_title('Return Scatter: Correlation Changes Across Regimes', fontweight='bold')
axes[1].axhline(y=0, color='gray', linewidth=0.5)
axes[1].axvline(x=0, color='gray', linewidth=0.5)
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3)

plt.tight_layout()
plt.show()

print("\nCorrelation by period:")
for name, data, _ in periods:
    print(f"  {name}: rho = {data.corr().iloc[0, 1]:.3f}")
print("\n-> Correlations are clearly NOT constant!")

---

## Exercise 2: CCC vs DCC Estimation on Real Data

**Objective:** Estimate CCC and DCC models and compare them.

**Tasks:**
1. Estimate univariate GARCH for S&P 500 and FTSE 100
2. Compute CCC correlation from standardized residuals
3. Estimate DCC parameters (a, b)
4. Test CCC vs DCC with likelihood ratio test

[QuantLet: TSA_ch5b_ccc_vs_dcc](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_ccc_vs_dcc)

In [ ]:
# Task 1: Univariate GARCH estimation
print("Step 1: Univariate GARCH(1,1) with Student-t")
print("=" * 50)

res_sp = arch_model(returns['SP500'].values, vol='Garch', p=1, q=1, dist='t').fit(disp='off')
res_ftse = arch_model(returns['FTSE100'].values, vol='Garch', p=1, q=1, dist='t').fit(disp='off')

for name, res in [('S&P 500', res_sp), ('FTSE 100', res_ftse)]:
    pers = res.params['alpha[1]'] + res.params['beta[1]']
    print(f"\n{name}: alpha={res.params['alpha[1]']:.4f}, beta={res.params['beta[1]']:.4f}, "
          f"persistence={pers:.4f}, nu={res.params['nu']:.2f}")

In [ ]:
# Task 2: CCC correlation
z_sp = res_sp.std_resid
z_ftse = res_ftse.std_resid
z = np.column_stack([z_sp, z_ftse])
Qbar = z.T @ z / len(z)

rho_ccc = Qbar[0, 1] / np.sqrt(Qbar[0, 0] * Qbar[1, 1])
print(f"CCC constant correlation: {rho_ccc:.4f}")

# Task 3: DCC estimation
a_hat, b_hat = estimate_dcc(z, Qbar)
print(f"\nDCC parameters: a = {a_hat:.4f}, b = {b_hat:.4f}, a+b = {a_hat+b_hat:.4f}")
print(f"Half-life: {np.log(0.5)/np.log(a_hat + b_hat):.1f} days")

# Get DCC correlations
Rt_dcc = dcc_correlations(z, Qbar, a_hat, b_hat)
rho_dcc = Rt_dcc[:, 0, 1]

print(f"\nDCC correlation range: [{rho_dcc.min():.3f}, {rho_dcc.max():.3f}]")
print(f"DCC mean correlation: {rho_dcc.mean():.3f}")

In [ ]:
# Task 4: Visualize and compare
rolling_corr = returns['SP500'].rolling(60).corr(returns['FTSE100'])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(returns.index, rho_dcc, color=COLORS['blue'], linewidth=0.8, label='DCC')
ax.plot(returns.index, rolling_corr, color=COLORS['orange'], linewidth=0.5, alpha=0.5, label='Rolling 60d')
ax.axhline(y=rho_ccc, color=COLORS['red'], linestyle='--', linewidth=1.5,
           label=f'CCC = {rho_ccc:.3f}')
ax.set_ylabel('Correlation')
ax.set_xlabel('Date')
ax.set_title('S&P 500 — FTSE 100: DCC vs CCC Correlation', fontweight='bold')
ax.set_ylim(-0.1, 1.0)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)

crisis_periods = [('2008-09-01', '2009-03-31'), ('2020-02-15', '2020-04-30')]
for start, end in crisis_periods:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color=COLORS['red'])

plt.tight_layout()
plt.show()

print("DCC captures the crisis spikes in correlation that CCC completely misses!")

---

## Exercise 3: Portfolio VaR with Dynamic Covariance

**Objective:** Calculate and backtest portfolio VaR using CCC and DCC.

**Tasks:**
1. Build portfolio variance using both models
2. Calculate 1-day VaR at 99%
3. Backtest both VaR models

[QuantLet: TSA_ch5b_var_backtest](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_var_backtest)

In [ ]:
# Task 1: Portfolio variance with DCC and CCC
sigma_sp = res_sp.conditional_volatility / 100
sigma_ftse = res_ftse.conditional_volatility / 100
w = np.array([0.5, 0.5])

h_11 = sigma_sp ** 2
h_22 = sigma_ftse ** 2
h_12_dcc = rho_dcc * sigma_sp * sigma_ftse
h_12_ccc = rho_ccc * sigma_sp * sigma_ftse

port_var_dcc = w[0]**2 * h_11 + w[1]**2 * h_22 + 2 * w[0] * w[1] * h_12_dcc
port_var_ccc = w[0]**2 * h_11 + w[1]**2 * h_22 + 2 * w[0] * w[1] * h_12_ccc

port_vol_dcc = np.sqrt(port_var_dcc)
port_vol_ccc = np.sqrt(port_var_ccc)

# Task 2: VaR 99%
z_99 = stats.norm.ppf(0.99)
VaR_dcc = z_99 * port_vol_dcc
VaR_ccc = z_99 * port_vol_ccc

# Portfolio returns
port_ret = (returns['SP500'] * w[0] + returns['FTSE100'] * w[1]) / 100

print("Portfolio VaR Comparison (99%, 1-day)")
print("=" * 50)
print(f"Average DCC VaR: {VaR_dcc.mean()*100:.3f}%")
print(f"Average CCC VaR: {VaR_ccc.mean()*100:.3f}%")

In [ ]:
# Task 3: Backtest VaR
violations_dcc = port_ret < -VaR_dcc
violations_ccc = port_ret < -VaR_ccc

vr_dcc = violations_dcc.mean()
vr_ccc = violations_ccc.mean()

print("VaR Backtest Results (expected: 1.00%)")
print("=" * 50)
print(f"DCC: {vr_dcc*100:.2f}% violations ({int(violations_dcc.sum())} out of {len(violations_dcc)})")
print(f"CCC: {vr_ccc*100:.2f}% violations ({int(violations_ccc.sum())} out of {len(violations_ccc)})")

# Kupiec test
for name, vr, n_viol in [('DCC', vr_dcc, int(violations_dcc.sum())),
                          ('CCC', vr_ccc, int(violations_ccc.sum()))]:
    n = len(port_ret)
    if 0 < vr < 1:
        LR = 2 * (n_viol * np.log(vr/0.01) + (n - n_viol) * np.log((1-vr)/(1-0.01)))
        pval = 1 - stats.chi2.cdf(abs(LR), 1)
        result = 'PASS' if pval > 0.05 else 'FAIL'
        print(f"  {name} Kupiec test: LR={LR:.2f}, p={pval:.4f} → {result}")

# Visualize
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(returns.index, port_ret * 100, color=COLORS['gray'], linewidth=0.4, alpha=0.6, label='Portfolio return')
ax.plot(returns.index, -VaR_dcc * 100, color=COLORS['blue'], linewidth=0.8, label='DCC VaR 99%')
ax.plot(returns.index, -VaR_ccc * 100, color=COLORS['red'], linewidth=0.8, alpha=0.6, label='CCC VaR 99%')

# Mark violations
viol_dates_dcc = returns.index[violations_dcc]
viol_rets = port_ret[violations_dcc] * 100
ax.scatter(viol_dates_dcc, viol_rets, color=COLORS['orange'], s=15, zorder=5, label='DCC violations')

ax.set_ylabel('Return / VaR (%)')
ax.set_xlabel('Date')
ax.set_title('VaR Backtest: DCC vs CCC (99%, 1-day)', fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4)

plt.tight_layout()
plt.show()

---

## Exercise 4: Dynamic Hedging Ratios

**Objective:** Compare CCC and DCC hedging ratios using S&P 500 and E-mini futures proxy.

**Tasks:**
1. Compute hedge ratios from both models
2. Compare hedging effectiveness
3. Analyze hedging during crisis periods

[QuantLet: TSA_ch5b_dynamic_hedge](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_dynamic_hedge)

In [ ]:
# Task 1: Hedge ratios (S&P 500 hedged with FTSE as diversifier)
hedge_dcc = h_12_dcc / h_22  # DCC hedge ratio
hedge_ccc = h_12_ccc / h_22  # CCC hedge ratio

print("Dynamic Hedge Ratios")
print("=" * 50)
print(f"DCC hedge ratio: mean={hedge_dcc.mean():.3f}, std={hedge_dcc.std():.3f}")
print(f"CCC hedge ratio: mean={hedge_ccc.mean():.3f}, std={hedge_ccc.std():.3f}")

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

axes[0].plot(returns.index, hedge_dcc, color=COLORS['blue'], linewidth=0.8, label='DCC')
axes[0].plot(returns.index, hedge_ccc, color=COLORS['red'], linewidth=0.8, alpha=0.6, label='CCC')
axes[0].set_ylabel('Hedge Ratio')
axes[0].set_title('Dynamic Hedge Ratio: DCC vs CCC', fontweight='bold')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)

# Task 2: Hedging effectiveness over time
# Rolling hedging effectiveness (60-day windows)
window = 60
he_dcc = np.full(len(returns), np.nan)
he_ccc = np.full(len(returns), np.nan)

for t in range(window, len(returns)):
    r_spot = returns['SP500'].values[t-window:t] / 100
    r_hedge = returns['FTSE100'].values[t-window:t] / 100
    h_dcc_t = hedge_dcc[t-window:t]
    h_ccc_t = hedge_ccc[t-window:t]
    
    var_unhedged = np.var(r_spot)
    var_hedged_dcc = np.var(r_spot - h_dcc_t * r_hedge)
    var_hedged_ccc = np.var(r_spot - h_ccc_t * r_hedge)
    
    he_dcc[t] = 1 - var_hedged_dcc / var_unhedged if var_unhedged > 0 else np.nan
    he_ccc[t] = 1 - var_hedged_ccc / var_unhedged if var_unhedged > 0 else np.nan

axes[1].plot(returns.index, he_dcc, color=COLORS['blue'], linewidth=0.8, label='DCC HE')
axes[1].plot(returns.index, he_ccc, color=COLORS['red'], linewidth=0.8, alpha=0.6, label='CCC HE')
axes[1].set_ylabel('Hedging Effectiveness')
axes[1].set_xlabel('Date')
axes[1].set_title('Rolling 60-Day Hedging Effectiveness (1 = perfect hedge)', fontweight='bold')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)

for start, end in crisis_periods:
    for ax in axes:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color=COLORS['red'])

plt.tight_layout()
plt.show()

print(f"\nMean hedging effectiveness: DCC={np.nanmean(he_dcc):.3f}, CCC={np.nanmean(he_ccc):.3f}")

---

## Exercise 5: Bitcoin-Ethereum Multivariate Analysis

**Objective:** Analyze crypto-to-crypto correlations and compare with equity-equity.

**Tasks:**
1. Download Bitcoin and Ethereum data
2. Estimate DCC-GARCH
3. Compare correlation dynamics with S&P 500 — FTSE 100

[QuantLet: TSA_ch5b_crypto_dcc](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_crypto_dcc)

In [ ]:
# Task 1: Download Bitcoin and Ethereum
print("Downloading Bitcoin and Ethereum data...")

btc = yf.download('BTC-USD', start='2017-01-01', end='2024-12-31', progress=False)
eth = yf.download('ETH-USD', start='2017-01-01', end='2024-12-31', progress=False)

r_btc = (btc['Close'].squeeze().pct_change() * 100).dropna()
r_eth = (eth['Close'].squeeze().pct_change() * 100).dropna()

crypto = pd.concat([r_btc, r_eth], axis=1, join='inner').dropna()
crypto.columns = ['Bitcoin', 'Ethereum']

print(f"\nCrypto Data: {crypto.index[0].date()} to {crypto.index[-1].date()}, {len(crypto)} obs")
print(f"\nDescriptive Statistics:")
print(crypto.describe().round(3))
print(f"\nFull-sample correlation: {crypto.corr().iloc[0, 1]:.4f}")

In [ ]:
# Task 2: Estimate DCC-GARCH for crypto
print("Estimating DCC-GARCH for Bitcoin-Ethereum...")

res_btc = arch_model(crypto['Bitcoin'].values, vol='Garch', p=1, q=1, dist='t').fit(disp='off')
res_eth = arch_model(crypto['Ethereum'].values, vol='Garch', p=1, q=1, dist='t').fit(disp='off')

z_btc = res_btc.std_resid
z_eth = res_eth.std_resid
z_crypto = np.column_stack([z_btc, z_eth])
Qbar_crypto = z_crypto.T @ z_crypto / len(z_crypto)

a_crypto, b_crypto = estimate_dcc(z_crypto, Qbar_crypto)
Rt_crypto = dcc_correlations(z_crypto, Qbar_crypto, a_crypto, b_crypto)
rho_crypto = Rt_crypto[:, 0, 1]

print(f"\nBitcoin GARCH: alpha={res_btc.params['alpha[1]']:.4f}, beta={res_btc.params['beta[1]']:.4f}")
print(f"Ethereum GARCH: alpha={res_eth.params['alpha[1]']:.4f}, beta={res_eth.params['beta[1]']:.4f}")
print(f"\nDCC parameters: a={a_crypto:.4f}, b={b_crypto:.4f}")
print(f"DCC correlation range: [{rho_crypto.min():.3f}, {rho_crypto.max():.3f}]")

In [ ]:
# Task 3: Compare crypto vs equity DCC correlations
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Crypto correlations
axes[0].plot(crypto.index, rho_crypto, color=COLORS['orange'], linewidth=0.8, label='DCC')
rolling_crypto = crypto['Bitcoin'].rolling(60).corr(crypto['Ethereum'])
axes[0].plot(crypto.index, rolling_crypto, color=COLORS['gray'], linewidth=0.5, alpha=0.5, label='Rolling 60d')
rho_ccc_crypto = Qbar_crypto[0, 1] / np.sqrt(Qbar_crypto[0, 0] * Qbar_crypto[1, 1])
axes[0].axhline(y=rho_ccc_crypto, color=COLORS['red'], linestyle='--', linewidth=1,
                label=f'CCC = {rho_ccc_crypto:.3f}')
axes[0].set_ylabel('Correlation')
axes[0].set_title('Bitcoin — Ethereum: DCC Correlation', fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=3)

# Comparison: distribution of DCC correlations
axes[1].hist(rho_dcc[rho_dcc != 0], bins=60, density=True, alpha=0.6, color=COLORS['blue'],
             edgecolor='white', label='S&P 500 — FTSE 100')
axes[1].hist(rho_crypto[rho_crypto != 0], bins=60, density=True, alpha=0.6, color=COLORS['orange'],
             edgecolor='white', label='Bitcoin — Ethereum')
axes[1].set_xlabel('DCC Correlation')
axes[1].set_ylabel('Density')
axes[1].set_title('Distribution of DCC Correlations: Equities vs Crypto', fontweight='bold')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)

plt.tight_layout()
plt.show()

print(f"\nComparison:")
print(f"  {'Metric':<30} {'Equities':>12} {'Crypto':>12}")
print(f"  {'-'*55}")
print(f"  {'DCC mean correlation':<30} {rho_dcc[rho_dcc!=0].mean():>12.3f} {rho_crypto[rho_crypto!=0].mean():>12.3f}")
print(f"  {'DCC std correlation':<30} {rho_dcc[rho_dcc!=0].std():>12.3f} {rho_crypto[rho_crypto!=0].std():>12.3f}")
print(f"  {'DCC min':<30} {rho_dcc[rho_dcc!=0].min():>12.3f} {rho_crypto[rho_crypto!=0].min():>12.3f}")
print(f"  {'DCC max':<30} {rho_dcc[rho_dcc!=0].max():>12.3f} {rho_crypto[rho_crypto!=0].max():>12.3f}")
print(f"  {'a (news reaction)':<30} {a_hat:>12.4f} {a_crypto:>12.4f}")
print(f"  {'b (persistence)':<30} {b_hat:>12.4f} {b_crypto:>12.4f}")

---

## Summary

### What We Learned (Using Real Financial Data)

1. **Correlations are NOT constant** — they spike during crises
   - S&P 500 — FTSE 100 correlation increased significantly during 2008 and COVID
   - Bitcoin — Ethereum correlation is high but also time-varying

2. **DCC outperforms CCC** for risk management
   - CCC underestimates portfolio risk during crises
   - DCC produces more accurate VaR (fewer violations in backtesting)

3. **Dynamic hedging** benefits from DCC
   - DCC hedge ratios adapt to changing correlations
   - CCC under-hedges during volatile periods

4. **Crypto vs equities**: different correlation dynamics
   - Crypto pairs tend to have higher but more variable correlations
   - Higher DCC `a` parameter = faster reaction to news

### Practical Workflow
1. Download data and align on common dates
2. Estimate univariate GARCH(1,1) with Student-t for each asset
3. Compute standardized residuals
4. Estimate DCC parameters (a, b) — only 2 parameters regardless of N!
5. Build $\mathbf{H}_t = \mathbf{D}_t \mathbf{R}_t \mathbf{D}_t$
6. Apply: portfolio VaR, hedge ratios, minimum variance weights
7. Backtest and validate